# 模型 6：BiLSTM + Attention

**负责人：** 6 号　**阶段：** 3 模型

token 序列 → BiLSTM + 加性注意力。

## 任务
- [x] 使用 FastText token 序列（按折版本）
- [x] 1–2 层 BiLSTM，hidden 128，处理 padding mask
- [x] 挑几条推文画注意力热力图
- [x] 每次实验先用 `new_run_id()`

## 产出
- `results/m6_bilstm.csv`
- 用 `save_preds` 保存 OOF 和测试集概率
- `figures/m6_attention_*.png`

第一版使用 Member 2 的 FastText token 序列。Member 5 的冻结 SBERT token 序列可在 11/8 后直接替换，无需改模型结构。

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED, N_FOLDS
from src.data import load_clean, load_splits
from src.evaluate import evaluate, new_run_id, save_result, plot_confusion
from src.cache import load_embedding, save_preds

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("仓库根目录:", ROOT)

In [ ]:
clean = load_clean()
splits = load_splits(clean=clean)
label_of = dict(zip(clean["id"], clean["label"]))
dev = splits[splits["split"] == "train"].reset_index(drop=True)
test = splits[splits["split"] == "test"].reset_index(drop=True)
y_dev = dev["id"].map(label_of).to_numpy()
y_test = test["id"].map(label_of).to_numpy()
print(f"Train {len(dev)}, Test {len(test)}")

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, sequences, masks, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.bool)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.sequences[idx], self.masks[idx], self.labels[idx]


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, h, mask):
        # h: (B, L, H), mask: (B, L)
        score = self.v(torch.tanh(self.W(h))).squeeze(-1)  # (B, L)
        score = score.masked_fill(~mask, -1e9)
        alpha = torch.softmax(score, dim=1)  # (B, L)
        context = torch.bmm(alpha.unsqueeze(1), h).squeeze(1)  # (B, H)
        return context, alpha


class BiLSTMAttention(nn.Module):
    def __init__(self, input_dim=100, hidden=128, n_layers=2, n_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=n_layers,
                            bidirectional=True, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.attn = AdditiveAttention(hidden * 2)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, n_classes)
        )
    def forward(self, x, mask):
        lengths = mask.sum(dim=1).cpu()
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(out, batch_first=True)  # (B, L, 2H)
        context, alpha = self.attn(h, mask)
        logits = self.fc(context)
        return logits, alpha

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, mask, y in loader:
        x, mask, y = x.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x, mask)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def predict_proba(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for x, mask, _ in loader:
            x, mask = x.to(device), mask.to(device)
            logits, _ = model(x, mask)
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(probs)

In [ ]:
def run_fold(fold, epochs=8, batch_size=32, lr=1e-3):
    # Load fold-specific FastText token sequences
    train_ids = dev.loc[dev["fold"] != fold, "id"].to_numpy()
    val_ids = dev.loc[dev["fold"] == fold, "id"].to_numpy()
    
    seq_tr = load_embedding(f"fasttext_tok_fold{fold}", ids=train_ids)
    mask_tr = load_embedding(f"fasttext_tok_fold{fold}_mask", ids=train_ids)
    seq_va = load_embedding(f"fasttext_tok_fold{fold}", ids=val_ids)
    mask_va = load_embedding(f"fasttext_tok_fold{fold}_mask", ids=val_ids)
    
    y_tr = y_dev[dev["fold"] != fold]
    y_va = y_dev[dev["fold"] == fold]
    
    train_ds = TweetDataset(seq_tr, mask_tr, y_tr)
    val_ds = TweetDataset(seq_va, mask_va, y_va)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    
    model = BiLSTMAttention().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    best_state = None
    best_val_loss = float("inf")
    for ep in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, mask, y in val_loader:
                x, mask, y = x.to(device), mask.to(device), y.to(device)
                logits, _ = model(x, mask)
                val_loss += criterion(logits, y).item() * len(y)
        val_loss /= len(val_ds)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    model.load_state_dict(best_state)
    val_proba = predict_proba(model, val_loader)
    return val_proba, model

In [ ]:
# Cross-validation
oof_proba = np.zeros((len(dev), 3))
for k in range(N_FOLDS):
    print(f"Fold {k} ...")
    val_proba, _ = run_fold(k)
    in_val = (dev["fold"] == k).to_numpy()
    oof_proba[in_val] = val_proba

oof_pred = oof_proba.argmax(axis=1)
run_id = new_run_id("bilstm", "fasttext_tok")
row = evaluate(y_dev, oof_pred, run_id=run_id, model="bilstm",
               embedding="fasttext_tok", split="cv",
               params={"hidden": 128, "layers": 2, "epochs": 8})
save_result(row, "m6_bilstm")
print("CV Macro-F1:", row["macro_f1"])

In [ ]:
# Final model on full training set
seq_full_tr = load_embedding("fasttext_tok_full", ids=dev["id"].to_numpy())
mask_full_tr = load_embedding("fasttext_tok_full_mask", ids=dev["id"].to_numpy())
seq_te = load_embedding("fasttext_tok_full", ids=test["id"].to_numpy())
mask_te = load_embedding("fasttext_tok_full_mask", ids=test["id"].to_numpy())

full_ds = TweetDataset(seq_full_tr, mask_full_tr, y_dev)
full_loader = DataLoader(full_ds, batch_size=32, shuffle=True)
test_ds = TweetDataset(seq_te, mask_te, y_test)
test_loader = DataLoader(test_ds, batch_size=32)

final_model = BiLSTMAttention().to(device)
optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
for ep in range(8):
    train_one_epoch(final_model, full_loader, optimizer, criterion)

test_proba = predict_proba(final_model, test_loader)
test_pred = test_proba.argmax(axis=1)
test_row = evaluate(y_test, test_pred, run_id=run_id, model="bilstm",
                    embedding="fasttext_tok", split="test",
                    params={"hidden": 128, "layers": 2, "epochs": 8})
save_result(test_row, "m6_bilstm")
print("Test Macro-F1:", test_row["macro_f1"])

save_preds(run_id, oof_ids=dev["id"].to_numpy(), oof_proba=oof_proba,
           test_ids=test["id"].to_numpy(), test_proba=test_proba, classes=[0,1,2])

plot_confusion(y_test, test_pred, "BiLSTM + Attention (test)",
               save_as=FIGURES_DIR / "m6_bilstm_confusion.png")

In [ ]:
# Attention heatmap examples
final_model.eval()
examples = [0, 10, 50]  # a few indices from test set
with torch.no_grad():
    for idx in examples:
        x = torch.tensor(seq_te[idx:idx+1], dtype=torch.float32).to(device)
        mask = torch.tensor(mask_te[idx:idx+1], dtype=torch.bool).to(device)
        _, alpha = final_model(x, mask)
        alpha = alpha.cpu().numpy()[0]
        length = int(mask.sum())
        fig, ax = plt.subplots(figsize=(8, 1.5))
        ax.imshow(alpha[:length].reshape(1, -1), cmap="Blues", aspect="auto")
        ax.set_title(f"Attention heatmap – test idx {idx}")
        ax.set_yticks([])
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / f"m6_attention_{idx}.png", dpi=200)
        plt.show()